#### PCA (주성분 분석)
- 차원 축소 기법
    - 피쳐의 개수가 줄어든다. 
- 고차원 데이터(다수의 피쳐를 가진 데이터셋)를 상관관계가 없는 새로운 축으로 변환 
- 데이터의 분산을 최대한 보존하면서 차원을 줄여서 시각화 
- parameter
    - n_components
        - 기본값 : None
        - 주성분의 개수 또는 비율로 설정 
        - None : 모든 주성분을 유지 
        - int : 선택할 주성분의 개수 
        - float(0~1) : 누적 설명 분산 비율을 기준으로 필요한 주성분의 개수를 선택 
        - 'mle' -> 자동으로 최적 차원 수 측정 (샘플의 개수 < 피려의 개수 인 경우메나 사용이 가능)
    - whiten
        - 기본값 : False
        - 주성분 벡터를 단위 분산으로 정규화를 할 것인가 지정 
        - 데이터의 정규화 효과
    - svd_solver
        - 기본값 :  'auto'
        - 분해 방식 지정 
        - 'auto' : 데이터의 크기에 따라서 자동 선택 
        - 'full' : 정확한 SVD 계산(적은 데이터에서 사용)
        - 'randomized' : 확률적 근사 알고리즘 ( 대규모 데이터에서 유용 )
- 속성
    - components_
        - 선택된 주성분 벡터 
    - explained_variance_
        - 각 주성분이 설명하는 분산 값
    - explained_vatiance_ratio_
        - 각 주성분이 설명하는 분산의 비율
        - 합계가 1에 가까울수록 원래 분산을 잘 보존 
    - singular_values_
        - 선택된 주성분에 대응하는 특이값
    - noise_variance_
        - 유지하지 않은 차원에서의 분산 추정치
- 메서드 
    - fit(X, y)
        - 주성분을 학습 
    - tansform(X)
        - 학습된 주성분 축을 기준으로 하여 새로운 데이터 변환
    - fit_transform(X, y)
        - PCA 학습 + 변환
    - get_covaiance()
        - PCA로 추정된 공분산 행렬 변환
    - get_precision()
        - PCA로 추정된 정밀도 행렬 변환

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.metrics import classification_report, r2_score

In [ ]:
# iris 데이터 로드 
iris = pd.read_csv("../csv/iris.csv")
iris.head()

In [ ]:
x = iris.drop('species', axis=1)
y = iris['species']

In [ ]:
# PCA 객체 생성 
pca = PCA()

In [ ]:
X_pca = pca.fit_transform(x, y)

In [ ]:
pd.concat( [ x, pd.DataFrame(X_pca) ], axis=1 )

In [ ]:
# 각 성분 별 분산의 비율을 출력 
ex_var_ratio = pca.explained_variance_ratio_

In [ ]:
# 분산의 비율을 누적합 생성 
cum_var_ratio = np.cumsum(ex_var_ratio)
cum_var_ratio

In [ ]:
# 분산의 비율을 그래프 시각화
plt.figure(figsize = (8, 8))

plt.bar(
    range(1, len(ex_var_ratio) + 1), ex_var_ratio, alpha=0.5
)

plt.step(
    range(1, len(cum_var_ratio) + 1), cum_var_ratio, 
    color = 'red', where='mid'
)

plt.show()

- 막대 : 각 주성분이 설명하는 분산의 비율
- 선 : 누적 분산의 비율
- 일반적으로 누적 분산의 비율이 90% ~ 95%이상이 되는 지점까지의 주성분만 선택하는 경우가 다수 -> 차원 축소의 근거 


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

# KFold 생성 
cv = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=42
)

# Pipeline 생성 -> Scaler -> PCA -> SVC
pipe = Pipeline(
    [
        ('scaler', StandardScaler()), 
        ('pca', PCA(random_state=42)), 
        ('svc', SVC(random_state=42, probability=True))
    ]
)

# 최적의 파라미터를 찾기위한 파라미터의 조합 
params = {
    'pca__n_components' : [None, 2, 3], 
    "svc__C" : [0.1, 1, 10], 
    'svc__gamma' : ['scale', 'auto'], 
    'svc__kernel' : ['linear', 'rbf']
}

# GridSearchCV 객체를 생성
grid = GridSearchCV(
    estimator= pipe, 
    param_grid= params, 
    scoring= 'accuracy', 
    cv = cv, 
    n_jobs = -1, 
    verbose= 1, 
    refit = True
)

In [17]:
grid.fit(X_train, y_train)

Fitting 5 folds for each of 36 candidates, totalling 180 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'pca__n_components': [None, 2, ...], 'svc__C': [0.1, 1, ...], 'svc__gamma': ['scale', 'auto'], 'svc__kernel': ['linear', 'rbf']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the com

In [18]:
# 최적의 모델을 무엇인가? 최적의 파라미터는 무엇인가
print('Best estimator : ', grid.best_estimator_)
print('Best Parameter : ', grid.best_params_)
print(classification_report(
    y_test, grid.predict(X_test)
))

Best estimator :  Pipeline(steps=[('scaler', StandardScaler()), ('pca', PCA(random_state=42)),
                ('svc',
                 SVC(C=0.1, kernel='linear', probability=True,
                     random_state=42))])
Best Parameter :  {'pca__n_components': None, 'svc__C': 0.1, 'svc__gamma': 'scale', 'svc__kernel': 'linear'}
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30



In [22]:
pd.DataFrame(
    grid.cv_results_
).sort_values('mean_test_score', ascending=False)[ 
    ['mean_fit_time', 'params', 'mean_test_score']
]

,mean_fit_time,params,mean_test_score
0,0.012262,"{'pca__n_components': None, 'svc__C': 0.1, 'sv...",0.975000
2,0.012259,"{'pca__n_components': None, 'svc__C': 0.1, 'sv...",0.975000
6,0.009606,"{'pca__n_components': None, 'svc__C': 1, 'svc_...",0.975000
4,0.011352,"{'pca__n_components': None, 'svc__C': 1, 'svc_...",0.975000
24,0.011467,"{'pca__n_components': 3, 'svc__C': 0.1, 'svc__...",0.975000
26,0.008741,"{'pca__n_components': 3, 'svc__C': 0.1, 'svc__...",0.975000
28,0.008528,"{'pca__n_components': 3, 'svc__C': 1, 'svc__ga...",0.966667
29,0.013421,"{'pca__n_components': 3, 'svc__C': 1, 'svc__ga...",0.966667
8,0.013177,"{'pca__n_components': None, 'svc__C': 10, 'svc...",0.966667
9,0.009437,"{'pca__n_components': None, 'svc__C': 10, 'svc...",0.966667
